# Generate Sun and Moon templates for a LAT analysis

This notebook runs the **Fermitools Solar System Tools** using your selected LAT data.
It produces two **full-sky, energy-dependent intensity cubes**: `sun_template.fits`
(solar disk + extended solar IC combined) and `moon_template.fits`.
Their spectra and absolute intensities are retained; do **not** normalize their
spatial slices to unity. In the LAT fit, start with one multiplicative normalization
per cube, initially 1.

**Edit the configuration cell, then run in order in your Fermitools environment.**
No threeML likelihood is initialized here. Use the event file **after** your final
event selection and `gtmktime`, not unfiltered downloaded events. A single notebook
run handles one Fermipy component / event-type selection. Run separately for
components with different GTIs, zenith cuts, IRFs, or event types.

The long livetime calculations run sequentially in time chunks, write logs, and
resume completed work. On Palmetto, run in an allocated compute-node session with
adequate RAM and disk space, not the login node. Solar livetime cubes can be large.

**Validation status:** notebook structure, Python syntax, and helper logic were
checked during preparation. A real Fermitools run was not possible in the authoring
environment; the installed Fermitools version and your data will be checked below.

The notebook also writes `_astromodels.fits` compatibility copies and includes a cell for your current threeML fit. All editable file paths are grouped in the first code cell.

## 1. Paths and settings

Paths written in your YAML are resolved relative to `YAML_BASE_DIR` (by default the
YAML's parent). Set that override if Fermipy originally resolved relative paths
from another working directory. Supply the actual selected event file explicitly.
`SCFILE` may be a FITS file, an `@list.txt`, or a Python list of FITS filenames.
`LIVETIME_CUBE` is optional; use only a cube made with the same GTIs and livetime
zenith cut. Its GTIs will be checked; the notebook cannot infer every original cut.

The notebook reads `selection.emin/emax/evtype/zmax`, `gtlike.irfs`, and energy
binning from the YAML. Overrides support incomplete YAMLs. The exposure energy grid
is shared by all three exposure/template tools and includes an adjustable margin
for energy dispersion. It need not equal the observed count-bin grid exactly, but
must cover all true energies used by your likelihood.

| Setting | File to supply |
|---|---|
| `CONFIG_YAML` | Your Fermipy YAML configuration |
| `EVENT_FILE` | Final selected, GTI-filtered LAT event FITS (often `ft1_00.fits`) |
| `SCFILE` | Spacecraft/pointing FITS (FT2); a file list is also supported |
| `LIVETIME_CUBE` | Optional matching ordinary livetime cube (often `ltcube_00.fits`) |

You do not need source maps or a counts cube. The profile files download automatically.


In [ ]:
from pathlib import Path

CONFIG_YAML = Path('/path/to/config.yaml')
EVENT_FILE = Path('/path/to/selected_events.fits')
SCFILE = '/path/to/spacecraft.fits'  # Or None to use data.scfile from the YAML
LIVETIME_CUBE = None          # Optional existing ordinary (not Sun/Moon) livetime cube
YAML_BASE_DIR = None          # None -> CONFIG_YAML.parent
COMPONENT = None             # None for single-component YAML; index or name otherwise
OUTDIR = Path('solar_lunar_templates')

### Advanced settings

The defaults below can usually be kept. Analysis-specific selections are read from your YAML.

In [ ]:
# None -> read from YAML. IRFS must be explicit, e.g. P8R3_SOURCE_V3.
IRFS_OVERRIDE = None
EVTYPE_OVERRIDE = None
LIVETIME_ZMAX_OVERRIDE = None # Match the ordinary LAT exposure's zenith prescription
EMIN_OVERRIDE = None         # MeV, measured analysis band
EMAX_OVERRIDE = None

EXPOSURE_BINS_PER_DECADE = 10
ENERGY_PADDING_BINS = None   # Auto: |edisp_bins|+2 if edisp=True, otherwise 0
LIVETIME_BINSZ = 0.5         # Keep 0.5 deg unless you have validated another choice
DCOSTHETA = 0.025
SUN_CHUNK_DAYS = 30
MOON_CHUNK_DAYS = 180
MAP_BINSZ = 0.5              # Full-sky Galactic CAR grid (720 x 360 by default)
PREVIEW_ENERGY_MEV = 1000.

# Optional alternative *intensity profile* files; NOT precomputed sky templates.
SUN_PROFILE = None
MOON_PROFILE = None
# Otherwise download the FSSC tutorial's baseline profiles automatically.

## 2. Imports, configuration, and executable checks

Dependencies: Fermitools with CALDB/IRFs initialized, plus `numpy`, `astropy`,
`matplotlib`, and `pyyaml`. These are commonly available in a Fermipy environment.
The notebook uses subprocess argument lists rather than shell command strings.

In [ ]:
import os, sys, json, copy, hashlib, shutil, subprocess, time, math
import urllib.request, importlib.metadata
import numpy as np
import yaml
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import astropy.units as u
import matplotlib.pyplot as plt

TOOLS = ['gtltcube', 'gtltcubesun', 'gtltsumsun', 'gtexpcube2', 'gtexphpsun', 'gtsuntemp']
missing = [t for t in TOOLS if shutil.which(t) is None]
if missing:
    raise RuntimeError('Activate your Fermitools environment and restart this kernel. Missing: '
                       + ', '.join(missing))

def merge_dict(base, extra):
    result = copy.deepcopy(base)
    for key, value in extra.items():
        if isinstance(value, dict) and isinstance(result.get(key), dict):
            result[key] = merge_dict(result[key], value)
        else:
            result[key] = copy.deepcopy(value)
    return result

CONFIG_YAML = CONFIG_YAML.expanduser().resolve()
base_dir = Path(YAML_BASE_DIR).expanduser().resolve() if YAML_BASE_DIR else CONFIG_YAML.parent
raw_cfg = yaml.safe_load(CONFIG_YAML.read_text())
if not isinstance(raw_cfg, dict):
    raise ValueError('Expected a Fermipy YAML mapping.')
base_cfg = {k: v for k, v in raw_cfg.items() if k != 'components'}
components = raw_cfg.get('components')
if components:
    if COMPONENT is None:
        raise ValueError('Set COMPONENT to the component index (list) or name (mapping).')
    selected = components[COMPONENT]
    if not isinstance(selected, dict):
        raise ValueError('Inline the selected component configuration in the YAML first.')
    cfg = merge_dict(base_cfg, selected)
else:
    cfg = base_cfg

def resolve_path(p):
    p = Path(os.path.expandvars(str(p))).expanduser()
    return (base_dir / p).resolve() if not p.is_absolute() else p.resolve()

EVENT_FILE = resolve_path(EVENT_FILE)
OUTDIR = OUTDIR.expanduser().resolve()
OUTDIR.mkdir(parents=True, exist_ok=True)
for name in ['profiles', 'chunks', 'logs', 'pfiles']:
    (OUTDIR / name).mkdir(exist_ok=True)
selection = cfg.get('selection', {})
gtlike = cfg.get('gtlike', {})
def choose(override, mapping, name):
    value = override if override is not None else mapping.get(name)
    if value is None:
        raise ValueError(f'Missing {name}: supply it in the YAML or configuration override.')
    return value
IRFS = str(choose(IRFS_OVERRIDE, gtlike, 'irfs'))
if IRFS.upper() == 'CALDB':
    raise ValueError('Set the explicit IRF name, rather than CALDB, for reproducibility.')
EVTYPE = int(choose(EVTYPE_OVERRIDE, selection, 'evtype'))
ZMAX = float(choose(LIVETIME_ZMAX_OVERRIDE, selection, 'zmax'))
EMIN = float(choose(EMIN_OVERRIDE, selection, 'emin'))
EMAX = float(choose(EMAX_OVERRIDE, selection, 'emax'))
if not (0 < EMIN < EMAX and 0 < ZMAX <= 180 and EVTYPE > 0):
    raise ValueError('Invalid energy, zenith, or event-type selection.')
if SUN_CHUNK_DAYS <= 0 or MOON_CHUNK_DAYS <= 0 or EXPOSURE_BINS_PER_DECADE <= 0:
    raise ValueError('Chunk durations and energy sampling must be positive.')

# Convert a spacecraft list into an explicit, absolute-path @list.
sc_input = SCFILE if SCFILE is not None else cfg.get('data', {}).get('scfile')
if sc_input is None:
    raise ValueError('Set SCFILE or data.scfile in the YAML.')
if isinstance(sc_input, (list, tuple)):
    sc_paths = [resolve_path(p) for p in sc_input]
elif str(sc_input).startswith('@'):
    listpath = resolve_path(str(sc_input)[1:])
    # Relative entries in an @list are resolved relative to the list file itself.
    sc_paths = []
    for line in listpath.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith('#'):
            p = Path(os.path.expandvars(line)).expanduser()
            sc_paths.append((listpath.parent / p).resolve() if not p.is_absolute() else p)
else:
    sc_paths = [resolve_path(sc_input)]
for p in [EVENT_FILE, *sc_paths]:
    if not p.is_file():
        raise FileNotFoundError(p)
if not sc_paths:
    raise ValueError('Spacecraft list is empty.')
if len(sc_paths) == 1:
    scarg = str(sc_paths[0])
else:
    sclist = OUTDIR / 'spacecraft_files.txt'
    sclist.write_text(''.join(str(p) + '\n' for p in sc_paths))
    scarg = '@' + str(sclist)

# Use a private writable parameter directory to avoid changing other jobs' pfiles.
pfiles = os.environ.get('PFILES', '')
if ';' not in pfiles:
    raise RuntimeError('PFILES is not initialized. Start Jupyter from the activated Fermitools environment.')
system_pfiles = pfiles.split(';', 1)[1]
TOOL_ENV = os.environ.copy()
TOOL_ENV['PFILES'] = str(OUTDIR / 'pfiles') + ';' + system_pfiles
TOOL_ENV.setdefault('OMP_NUM_THREADS', '1')
versions = {}
for pkg in ['fermitools', 'fermipy', 'astropy', 'numpy']:
    try: versions[pkg] = importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError: versions[pkg] = 'not reported by Python metadata'
print(json.dumps({'IRFS': IRFS, 'evtype': EVTYPE, 'livetime_zmax': ZMAX,
                  'energy_band_MeV': [EMIN, EMAX], 'event_file': str(EVENT_FILE),
                  'spacecraft_files': [str(p) for p in sc_paths], 'versions': versions}, indent=2))

## 3. GTIs, coverage, and energy grid

Review the printed event-file selection keywords against your YAML. In particular,
match event class/type and the zenith-angle prescription used for your existing LAT
exposure. The event-file GTIs are authoritative for the observing time; no fixed
mission dates are assumed. Spacecraft coverage is checked across every GTI.

In [ ]:
def merge_intervals(intervals):
    a = np.asarray(intervals, dtype=float).reshape(-1, 2)
    if not np.isfinite(a).all() or np.any(a[:, 1] <= a[:, 0]):
        raise ValueError('GTIs must be finite intervals with STOP > START.')
    result = []
    for start, stop in a[np.argsort(a[:, 0])]:
        if result and start <= result[-1][1] + 1e-6:
            result[-1][1] = max(result[-1][1], stop)
        else:
            result.append([float(start), float(stop)])
    return np.asarray(result, dtype=float).reshape(-1, 2)

def read_gti(path):
    with fits.open(path, memmap=True) as f:
        g = f['GTI'].data
        return merge_intervals(np.column_stack([g['START'], g['STOP']]))

def clipped_gti(gti, start, stop):
    a = np.maximum(gti[:, 0], start)
    b = np.minimum(gti[:, 1], stop)
    return np.column_stack([a[b > a], b[b > a]])

def check_gti(path, expected):
    actual = read_gti(path)
    expected = merge_intervals(expected)
    if actual.shape != expected.shape or not np.allclose(actual, expected, rtol=0, atol=1e-3):
        raise RuntimeError(f'GTI mismatch in {path}. Stop: do not sum overlapping or incorrectly timed cubes.')

GTI = read_gti(EVENT_FILE)
if not len(GTI):
    raise ValueError('Selected event file has no good-time intervals.')
TSTART, TSTOP = float(GTI[0, 0]), float(GTI[-1, 1])
with fits.open(EVENT_FILE) as f:
    h = f['EVENTS'].header
    dss = {k: h[k] for k in h if k.startswith(('DSTYP', 'DSVAL', 'DSREF', 'DSUNI'))}
print('Event selection keywords:', json.dumps(dss, indent=2))
print('MET range:', TSTART, TSTOP, '; accepted time [days]:', np.diff(GTI).sum()/86400)
sc_intervals = []
for p in sc_paths:
    with fits.open(p, memmap=True) as f:
        tab = f['SC_DATA'].data
        sc_intervals.extend(zip(tab['START'], tab['STOP']))
sc_coverage = merge_intervals(sc_intervals)
j = 0
for start, stop in GTI:
    while j < len(sc_coverage) and sc_coverage[j, 1] < start - 1e-3:
        j += 1
    if j == len(sc_coverage) or sc_coverage[j, 0] > start + 1e-3 or sc_coverage[j, 1] < stop - 1e-3:
        raise ValueError(f'Spacecraft history does not fully cover GTI {start}:{stop}.')

analysis_bpd = float(cfg.get('binning', {}).get('binsperdec', 8))
padding = ENERGY_PADDING_BINS
if padding is None:
    padding = abs(int(gtlike.get('edisp_bins', -1))) + 2 if gtlike.get('edisp', False) else 0
if padding < 0:
    raise ValueError('ENERGY_PADDING_BINS must be nonnegative.')
ELO = EMIN / 10**(padding/analysis_bpd)
EHI = EMAX * 10**(padding/analysis_bpd)
NBINS = int(math.ceil(np.log10(EHI/ELO) * EXPOSURE_BINS_PER_DECADE))
energy_pars = dict(ebinalg='LOG', emin=ELO, emax=EHI, enumbins=NBINS,
                   bincalc='EDGE', ebinfile='NONE')
expected_energy = np.geomspace(ELO, EHI, NBINS+1)
nx, ny = int(round(360/MAP_BINSZ)), int(round(180/MAP_BINSZ))
if not np.isclose(nx*MAP_BINSZ, 360) or not np.isclose(ny*MAP_BINSZ, 180):
    raise ValueError('MAP_BINSZ must tile 360 and 180 degrees exactly.')
map_pars = dict(cmap='none', nxpix=nx, nypix=ny, binsz=MAP_BINSZ,
                coordsys='GAL', xref=0., yref=0., axisrot=0., proj='CAR')
print('Template energy coverage [MeV]:', ELO, EHI, '; energy planes:', NBINS+1)
print('Full-sky map size:', nx, 'x', ny)

## 4. Baseline emission profiles

The defaults are the FSSC's downloadable `solar_profile_v2r0.fits` and
`lunar_profile_v2r0.fits`. The solar file combines the disk and solar IC. They are
**steady, historical baseline models**, with high-energy extrapolations, not
updated measurements across your entire observing interval. They omit solar flares
and solar-cycle variations. A fitted normalization is a first approximation; it
does not correct every possible spectral or time-dependent mismatch. You can
supply better-matched profiles in cell 1 without changing the pipeline.

In [ ]:
PROFILE_ROOT = 'https://fermi.gsfc.nasa.gov/ssc/data/analysis/scitools/data/latSolarTemplate/'

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(4*1024*1024), b''):
            h.update(block)
    return h.hexdigest()

def get_profile(override, name):
    target = resolve_path(override) if override is not None else OUTDIR/'profiles'/name
    if not target.exists() and override is None:
        temp = target.with_suffix('.download')
        try:
            request = urllib.request.Request(PROFILE_ROOT+name, headers={'User-Agent': 'LAT-template-notebook'})
            with urllib.request.urlopen(request, timeout=120) as response, open(temp, 'wb') as out:
                shutil.copyfileobj(response, out)
            with fits.open(temp) as f:
                f.verify('exception')
            temp.replace(target)
        except Exception:
            temp.unlink(missing_ok=True)
            raise RuntimeError(f'Download failed. Download {PROFILE_ROOT+name} manually and set its path in cell 1.')
    with fits.open(target) as f:
        for ext in ['ANGLES', 'ENERGIES', 'SST_PROFILE']:
            if ext not in f:
                raise ValueError(f'{target} is missing {ext}. Supply an SST intensity profile.')
        e = np.asarray(f['ENERGIES'].data.field(0), dtype=float).ravel()
        if ELO < e.min()*(1-1e-6) or EHI > e.max()*(1+1e-6):
            raise ValueError(f'Requested energy range exceeds {target.name}: {e.min()} to {e.max()} MeV. '
                             'Adjust padding/range or supply a suitable profile.')
    print(target.name, 'SHA256', sha256_file(target))
    return target
profiles = {'SUN': get_profile(SUN_PROFILE, 'solar_profile_v2r0.fits'),
            'MOON': get_profile(MOON_PROFILE, 'lunar_profile_v2r0.fits')}

## 5. Restartable command runner

A manifest protects this output directory against accidental reuse with changed
settings or input-file metadata. For a different analysis, choose a new `OUTDIR`.
Completed tasks have sidecar receipts; incomplete tasks are rerun. No source data
are modified. Tool output is written to log files, with a heartbeat while running.
File identity uses path, size, and modification time for large input data; do not
replace an input while preserving all three. Profile files are hashed.

In [ ]:
def file_stamp(p):
    p = Path(p).resolve(); st = p.stat()
    return {'path': str(p), 'size': st.st_size, 'mtime_ns': st.st_mtime_ns}

existing_lt = resolve_path(LIVETIME_CUBE) if LIVETIME_CUBE is not None else None
if existing_lt is not None:
    check_gti(existing_lt, GTI)
manifest = dict(schema=1, config=cfg, events=file_stamp(EVENT_FILE),
                spacecraft=[file_stamp(p) for p in sc_paths],
                existing_livetime=file_stamp(existing_lt) if existing_lt else None,
                irfs=IRFS, evtype=EVTYPE, zmax=ZMAX,
                energy=energy_pars, spatial=map_pars, versions=versions,
                lt_binsz=LIVETIME_BINSZ, dcostheta=DCOSTHETA,
                chunk_days={'SUN': SUN_CHUNK_DAYS, 'MOON': MOON_CHUNK_DAYS},
                profiles={b: sha256_file(p) for b,p in profiles.items()})
manifest_text = json.dumps(manifest, sort_keys=True, indent=2, default=str)
manifest_path = OUTDIR/'run_manifest.json'
if manifest_path.exists() and manifest_path.read_text() != manifest_text:
    raise RuntimeError('Inputs/settings changed. Choose a new OUTDIR to avoid reusing incompatible products.')
manifest_path.write_text(manifest_text)
run_key = hashlib.sha256(manifest_text.encode()).hexdigest()

def run_tool(tool, outfile, **parameters):
    outfile = Path(outfile)
    receipt = Path(str(outfile)+'.done.json')
    log = OUTDIR/'logs'/(outfile.name+'.log')
    parameters = dict(parameters, outfile=str(outfile), clobber='yes', chatter=2, mode='h')
    args = [shutil.which(tool)] + [f'{key}={value}' for key,value in parameters.items()]
    task_key = hashlib.sha256(json.dumps([run_key,args], sort_keys=True).encode()).hexdigest()
    if outfile.exists() and receipt.exists():
        saved = json.loads(receipt.read_text())
        if saved.get('key') == task_key and saved.get('output') == file_stamp(outfile):
            print('Reuse:', outfile.name, flush=True)
            return outfile
    receipt.unlink(missing_ok=True)
    print('Run:', tool, '->', outfile.name, flush=True)
    start = time.monotonic()
    with open(log, 'w') as stream:
        stream.write(json.dumps(args)+'\n'); stream.flush()
        process = subprocess.Popen(args, stdout=stream, stderr=subprocess.STDOUT, env=TOOL_ENV)
        try:
            while True:
                try:
                    status = process.wait(timeout=30)
                    break
                except subprocess.TimeoutExpired:
                    print(f'  {tool}: {(time.monotonic()-start)/60:.1f} min; log: {log.name}', flush=True)
        except BaseException:
            process.terminate()
            try: process.wait(timeout=10)
            except subprocess.TimeoutExpired: process.kill(); process.wait()
            raise
    if status != 0 or not outfile.is_file():
        print(log.read_text(errors='replace')[-6000:])
        raise RuntimeError(f'{tool} failed (status {status}). See {log}.')
    with fits.open(outfile, memmap=True) as f:
        f.verify('exception')
    receipt.write_text(json.dumps({'key': task_key, 'output': file_stamp(outfile), 'args': args}, indent=2))
    return outfile

## 6. Ordinary sky exposure

This is the denominator used to convert the moving-source prediction to an
average intensity on the stationary sky. It must use the same GTIs, IRFs, event
type, energy grid, and livetime zenith prescription as the moving-source exposure.

In [ ]:
ordinary_lt = existing_lt
if ordinary_lt is None:
    ordinary_lt = run_tool('gtltcube', OUTDIR/'ordinary_ltcube.fits',
                           evfile=str(EVENT_FILE), scfile=scarg,
                           dcostheta=DCOSTHETA, binsz=LIVETIME_BINSZ,
                           zmax=ZMAX, phibins=0, tmin=0, tmax=0)
check_gti(ordinary_lt, GTI)
average_exposure = run_tool('gtexpcube2', OUTDIR/'average_exposure.fits',
                            infile=str(ordinary_lt), irfs=IRFS, evtype=EVTYPE,
                            edisp_bins=0, ignorephi='no', thmin=0, thmax=180,
                            **map_pars, **energy_pars)

## 7. Moving-source livetime cubes — the expensive step

Chunks use disjoint time intervals and the original event GTIs. Empty intervals
are skipped. The GTIs written by each tool are checked against the expected clipped
GTIs before summing, preventing unnoticed time overlap. A later rerun reuses
finished chunks. The full Sun calculation uses 180-degree angular coverage; the
Moon uses 1 degree. Keep all chunk files until the run is validated.

In [ ]:
def time_chunks(gti, days):
    start, stop = gti[0,0], gti[-1,1]
    edges = np.arange(start, stop, days*86400.)
    edges = np.r_[edges, stop]
    for i, (lo,hi) in enumerate(zip(edges[:-1], edges[1:])):
        expected = clipped_gti(gti, lo, hi)
        if len(expected):
            yield i, float(lo), float(hi), expected

moving_lt = {}
for body, days, angle, power in [('SUN', SUN_CHUNK_DAYS, 180., 2.7),
                                 ('MOON', MOON_CHUNK_DAYS, 1., 2.)]:
    tasks = list(time_chunks(GTI, days))
    print(body, ':', len(tasks), 'nonempty time chunks')
    files = []
    for number, lo, hi, expected in tasks:
        path = OUTDIR/'chunks'/f'{body.lower()}_lt_{number:05d}.fits'
        run_tool('gtltcubesun', path, evfile=str(EVENT_FILE), scfile=scarg,
                 body=body, tmin=lo, tmax=hi, dcostheta=DCOSTHETA,
                 thetasunmax=angle, powerbinsun=power, binsz=LIVETIME_BINSZ,
                 phibins=0, zmax=ZMAX)
        check_gti(path, expected)
        files.append(path)
    if len(files) == 1:
        moving_lt[body] = files[0]
    else:
        listfile = OUTDIR/f'{body.lower()}_livetime_files.txt'
        listfile.write_text(''.join(str(f)+'\n' for f in files))
        moving_lt[body] = run_tool('gtltsumsun', OUTDIR/f'{body.lower()}_ltcube.fits',
                                   infile1='@'+str(listfile), infile2='none')
    check_gti(moving_lt[body], GTI)

## 8. Moving-source exposures and final intensity cubes

These outputs are diffuse intensity models, not source-count maps or PSF-convolved
predictions. The normal LAT response calculation must still be applied in the fit.

In [ ]:
templates = {}
for body in ['SUN', 'MOON']:
    exposure = run_tool('gtexphpsun', OUTDIR/f'{body.lower()}_moving_exposure.fits',
                         infile=str(moving_lt[body]), irfs=IRFS, evtype=EVTYPE,
                         binsz=0, ignorephi='no', thmin=0, thmax=180, **energy_pars)
    templates[body] = run_tool('gtsuntemp', OUTDIR/f'{body.lower()}_template.fits',
                              expsun=str(exposure), avgexp=str(average_exposure),
                              sunprof=str(profiles[body]), irfs=IRFS, evtype=EVTYPE,
                              **map_pars, **energy_pars)
print('Final templates:', {k:str(v) for k,v in templates.items()})

## 9. Validate the FITS products and inspect your GC ROI

The ENERGIES table is authoritative (MeV). We require the requested shared grid,
finite nonnegative intensity, and nonzero emission. The plots display the actual
cube planes at the nearest available energy. The cyan outline is the 40 x 40 degree
GC ROI. Zeros far from the Moon's track are expected.

In [ ]:
report = {}
fig, axes = plt.subplots(1,2,figsize=(14,5),layout='constrained')
from matplotlib.patches import Rectangle
for ax, body in zip(axes, ['SUN','MOON']):
    with fits.open(templates[body], memmap=True) as f:
        f.verify('exception')
        cube = f[0].data
        if cube is None or cube.ndim != 3:
            raise ValueError(f'{body}: expected a 3D primary-HDU mapcube.')
        energy = np.asarray(f['ENERGIES'].data.field(0), dtype=float).ravel()
        if len(energy) != len(cube) or energy.shape != expected_energy.shape:
            raise ValueError(f'{body}: unexpected energy-plane count.')
        if not np.allclose(energy, expected_energy, rtol=1e-5, atol=0):
            raise ValueError(f'{body}: output energy grid differs from requested grid.')
        if not np.isfinite(cube).all() or np.any(cube < 0) or not np.any(cube > 0):
            raise ValueError(f'{body}: invalid intensity values; inspect tool logs.')
        k = int(np.argmin(abs(np.log(energy/PREVIEW_ENERGY_MEV))))
        plane = cube[k].copy()
        w = WCS(f[0].header).celestial
        unit = f[0].header.get('BUNIT', 'ph cm^-2 s^-1 MeV^-1 sr^-1 (SST convention)')
        report[body] = dict(path=str(templates[body]), shape=list(cube.shape),
                            energy_min_MeV=float(energy[0]), energy_max_MeV=float(energy[-1]),
                            preview_energy_MeV=float(energy[k]), unit=unit)
    # Nearest-pixel display sampled on a Galactic grid, without changing the FITS.
    ll,bb = np.meshgrid(np.linspace(30,-30,481),np.linspace(-30,30,481))
    px,py = w.world_to_pixel(SkyCoord(l=ll*u.deg,b=bb*u.deg,frame='galactic'))
    ix,iy = np.rint(px).astype(int),np.rint(py).astype(int)
    valid = (ix>=0)&(ix<plane.shape[1])&(iy>=0)&(iy<plane.shape[0])
    display = np.full(ll.shape,np.nan)
    display[valid] = plane[iy[valid],ix[valid]]
    im = ax.imshow(display, origin='lower', extent=[30,-30,-30,30], cmap='magma',vmin=0)
    ax.add_patch(Rectangle((-20,-20),40,40,fill=False,edgecolor='cyan'))
    ax.set(xlabel='Galactic longitude [deg]',ylabel='Galactic latitude [deg]',
           title=f'{body}: {energy[k]/1000:.3g} GeV')
    fig.colorbar(im,ax=ax,label='ph cm⁻² s⁻¹ MeV⁻¹ sr⁻¹')
fig.savefig(OUTDIR/'sun_moon_preview.png',dpi=160)
plt.show()
(OUTDIR/'validation.json').write_text(json.dumps(report,indent=2))
print(json.dumps(report,indent=2))

## 10. Prepare copies for your astromodels `GalpropMap` loader

Your `diffuse-analysis` FermipyLike supports `GalpropMap` as a physical LAT
`MapCubeFunction`. It requires a **fixed `Constant(k=1)` spectrum** and fits the
map's **`spatial_shape.K`**. No plugin modification is needed for this route in the
branch inspected when this notebook was prepared (2026-10-13).

Your astromodels loader derives energies from the primary HDU's third-axis header.
Standard Fermitools cubes instead provide the authoritative values in the
`ENERGIES` extension. The cell below creates `_astromodels.fits` copies with an
explicit, regular **log10(E/MeV)** third-axis header. It leaves pixel intensities,
spatial WCS, and the ENERGIES table unchanged. Use these copies in your model.
This is a header compatibility conversion, not a flux or energy-unit conversion.

In [ ]:
def prepare_astromodels_cube(source, destination):
    with fits.open(source) as hdul:
        energy_hdu = hdul['ENERGIES']
        energy = np.asarray(energy_hdu.data.field(0), dtype=float).ravel()
        energy_unit = energy_hdu.columns[0].unit
        if energy_unit and u.Unit(energy_unit) != u.MeV:
            raise ValueError('Expected ENERGIES values in MeV; inspect the SST output.')
        if not np.all(energy > 0) or len(energy) != hdul[0].data.shape[0]:
            raise ValueError('Invalid energy axis.')
        loge = np.log10(energy)
        delta = (loge[-1]-loge[0])/(len(loge)-1)
        if not np.allclose(loge, loge[0]+np.arange(len(loge))*delta, rtol=0, atol=1e-6):
            raise ValueError('GalpropMap requires a regular grid: use LOG energy binning.')
        header = hdul[0].header
        header['CRPIX3'] = 1.
        header['CRVAL3'] = float(loge[0])
        header['CDELT3'] = float(delta)
        header['CTYPE3'] = 'LOG10E'
        header['CUNIT3'] = ''
        header['BUNIT'] = 'ph cm-2 s-1 MeV-1 sr-1'
        header.add_history('Third-axis header is log10(E/MeV) for astromodels GalpropMap.')
        header.add_history('Intensities, celestial WCS and ENERGIES table are unchanged.')
        hdul.writeto(destination, overwrite=True, checksum=True)
    with fits.open(source) as original, fits.open(destination, checksum=True) as converted:
        assert np.array_equal(original[0].data, converted[0].data)
        assert np.array_equal(original['ENERGIES'].data, converted['ENERGIES'].data)
        reconstructed = 10**(converted[0].header['CRVAL3'] + np.arange(len(energy))*converted[0].header['CDELT3'])
        assert np.allclose(reconstructed, energy, rtol=1e-5)
        converted.verify('exception')
    return Path(destination)

model_templates = {
    body: prepare_astromodels_cube(path, OUTDIR/f'{body.lower()}_template_astromodels.fits')
    for body, path in templates.items()
}
print({body:str(path) for body,path in model_templates.items()})

## 11. Copy this cell into your existing threeML fit notebook

Run **after creating `model` and before initializing the LAT likelihood**. Update
only `template_dir`. This cell is disabled here so this generation notebook can
run end-to-end without a threeML model; set `ADD_TO_EXISTING_MODEL = True` in your
fit notebook. It replaces only sources named `LAT_sun` / `LAT_moon` if already
present. Each foreground adds **one free normalization**, two in total.

`set_frame(ICRS())` sets the coordinates passed by threeML; the FITS remains in
Galactic coordinates. The adapter's logarithmic energy axis uses MeV, and threeML
energies use keV. `GalpropMap` performs the conversion.

For multiple Fermipy components with different event selections, this cell alone
is not sufficient to route different per-component cubes: it supplies a single
mapcube per source. Generate each component's cubes separately and verify the
plugin's per-component routing before a combined fit. This notebook/cell targets
your single-selection LAT fit.

In [ ]:
ADD_TO_EXISTING_MODEL = False  # Set True when pasted into your fit notebook.
if ADD_TO_EXISTING_MODEL:
    from pathlib import Path
    from astromodels import GalpropMap, Constant, ExtendedSource
    from astropy.coordinates import ICRS
    import astropy.units as u

    template_dir = Path('/path/to/solar_lunar_templates').expanduser().resolve()

    def make_solar_system_source(name, filename):
        path = template_dir / filename
        if not path.is_file():
            raise FileNotFoundError(path)
        spatial = GalpropMap()
        spatial.load_file(
            str(path), ihdu=0,
            map_is_e2_dnde=False,
            input_energy_unit=u.keV,
            fits_energy_unit=u.MeV,
            energy_axis_is_log10=True,
            use_region=False,
        )
        spatial.set_frame(ICRS())
        spatial.K = 1.0
        spatial.K.bounds = (0.0, 10.0)
        spatial.K.free = True

        spectrum = Constant()
        spectrum.k = 1.0
        spectrum.k.fix = True
        return ExtendedSource(name, spatial_shape=spatial, spectral_shape=spectrum)

    solar_sources = [
        make_solar_system_source('LAT_sun', 'sun_template_astromodels.fits'),
        make_solar_system_source('LAT_moon', 'moon_template_astromodels.fits'),
    ]
    for source in solar_sources:
        if source.name in model.sources:
            model.remove_source(source.name)
        model.add_source(source)

    # Reinitialize LAT / JointLikelihood after this cell.
    # Keep LAT_sun and LAT_moon excluded from your SPI/COSI plugins.
    # If these names already have cached source maps from DIFFERENT cubes,
    # regenerate those source maps or use a fresh LAT output directory.

## 12. Optional standard LAT XML definitions

This writes a two-source XML snippet using the original standard Fermitools
mapcubes. It is an alternative for direct Fermitools/Fermipy use, not an additional
step when using the threeML cell above. Do not add the sources twice. The cube
already contains the spectrum; the fixed index 0 gives a flat multiplier with
one free normalization per source. This XML is not a complete analysis model.

In [ ]:
import xml.etree.ElementTree as ET
root = ET.Element('source_library', title='Solar and lunar foregrounds')
for body in ['SUN','MOON']:
    source = ET.SubElement(root,'source',name='LAT_'+body.lower(),type='DiffuseSource')
    spectrum = ET.SubElement(source,'spectrum',type='PowerLaw')
    for name,value,minimum,maximum,free in [('Prefactor','1','0','10','1'),
                                           ('Index','0','-1','1','0'),
                                           ('Scale','1000','1','1000000','0')]:
        ET.SubElement(spectrum,'parameter',name=name,value=value,scale='1',
                      min=minimum,max=maximum,free=free)
    spatial = ET.SubElement(source,'spatialModel',type='MapCubeFunction',file=str(templates[body]))
    ET.SubElement(spatial,'parameter',name='Normalization',value='1',scale='1',min='0',max='10',free='0')
ET.indent(root)
xmlpath = OUTDIR/'sun_moon_sources.xml'
ET.ElementTree(root).write(xmlpath,encoding='utf-8',xml_declaration=True)
print(xmlpath)

## References and practical limits

- [FSSC solar/lunar template tutorial](https://fermi.gsfc.nasa.gov/ssc/data/analysis/scitools/solar_template.html)
- [Baseline profile definitions](https://fermi.gsfc.nasa.gov/ssc/data/analysis/scitools/SSTmodels_description.pdf)
- [Johannesson & Orlando 2013: Solar System Tools](https://arxiv.org/abs/1307.0197)
- [Fermitools SolarSystemTools parameter files](https://github.com/fermi-lat/SolarSystemTools/tree/master/pfiles)

All tools receive the same explicit energy grid and event type. Changes to the
observing selection require matching templates. Normalization 1 denotes the
chosen steady emission profile, not a unit-integral spatial template. The current
setup has two free normalizations; independently fitting solar disk and solar IC
would require separate input profiles and separate output cubes.

If a task fails, inspect its log, fix the cause, and rerun the cell. If inputs or
scientific settings changed, use a new output directory. If a chunk fails its GTI
check, stop and inspect the installed tool behavior before combining any cubes.